# HFS Project — Data Fetcher & Preprocessing Pipeline

**Project**: Modern Multi-Factor Market Neutral Equity Strategy for S&P 500

This notebook implements **Phase 1 (Data Acquisition & Preprocessing)** and builds the data foundation for **Phase 2 (Factor Engineering)** as specified in the project proposal.

### Key Design Principles
1. **No survivorship bias**: Full price history is retained for all stocks ever included in the S&P 500, including pre-entry and post-exit periods (needed for momentum lookback and rolling beta).
2. **Point-in-Time (PIT) discipline**: All fundamental data is aligned via `rdq` with a minimum 45-day delay from `datadate`, enforced by assertion checks.
3. **Intra-sector standardization**: All Z-scores are computed within `date × gsector` groups using MAD winsorization, consistent with Proposal Task 1.3.
4. **Formula consistency**: Value = 0.67·Z(EY) + 0.33·Z(P/B); Size = −Z(mktcap).

### Global Parameters

In [1]:
# ============================================================
# Cell 1: Imports & Global Parameters
# ============================================================
import wrds
import pandas as pd
import numpy as np

db = wrds.Connection()

# ---- Global parameters ----
BACKTEST_START = '2010-01-01'
BACKTEST_END   = '2024-12-31'

# Extended range for lookback windows (momentum 12m, rolling beta 63d, covariance 252d)
PRICE_START = '2008-01-01'

# Fundamentals buffer: need lagged quarters for PIT alignment at PRICE_START
FUNDQ_START = '2006-01-01'

# PIT filing delay: minimum days from datadate before data is considered "available"
# Proposal specifies 45–90 day filing delay from fiscal period end
MIN_DELAY_FROM_DATADATE = 45   # Lower bound per proposal

print(f"Backtest window:  {BACKTEST_START} → {BACKTEST_END}")
print(f"Price data from:  {PRICE_START}")
print(f"Fundq data from:  {FUNDQ_START}")

Loading library list...
Done
Backtest window:  2010-01-01 → 2024-12-31
Price data from:  2008-01-01
Fundq data from:  2006-01-01


---
## Phase 1: Data Acquisition & Preprocessing

### Task 1.1 — S&P 500 Constituent History & Dynamic Price Panel

In [2]:
# ============================================================
# Cell 2: S&P 500 Historical Constituent List
# ============================================================
# Source: crsp.dsp500list — official S&P 500 membership history

sp500_members = db.raw_sql(f"""
    SELECT permno, start AS start_date, ending AS end_date
    FROM crsp.dsp500list
    WHERE COALESCE(ending, '2099-12-31'::date) >= '{PRICE_START}'
""", date_cols=["start_date", "end_date"])

# Fill missing end dates (currently active members)
sp500_members["end_date"] = sp500_members["end_date"].fillna(pd.Timestamp("2099-12-31"))

print(f"S&P 500 membership records: {len(sp500_members):,}")
print(f"Unique permnos (ever in S&P 500): {sp500_members['permno'].nunique()}")
print(f"Date range: {sp500_members['start_date'].min().date()} → {sp500_members['end_date'].max().date()}")

S&P 500 membership records: 891
Unique permnos (ever in S&P 500): 871
Date range: 1925-12-31 → 2024-12-31


In [3]:
# ============================================================
# Cell 3: Daily Price / Volume / Return Data
# ============================================================
# Pull ALL trading days for ALL permnos that were ever in S&P 500,
# starting from PRICE_START to allow sufficient lookback for:
#   - Momentum factor: 12 months of prior returns
#   - Rolling Beta: 63 trading days
#   - Rolling covariance: 252 trading days
#
# CRITICAL FIX: Do NOT filter by in_sp500 here. The full price history
# is needed for lookback calculations. The in_sp500 flag is used later
# to define the tradable universe at portfolio construction time.

permnos = sp500_members['permno'].unique().tolist()
permnos_str = ','.join(str(p) for p in permnos)

price_raw = db.raw_sql(f"""
    SELECT
        permno,
        date,
        abs(prc)  AS prc,     -- CRSP convention: negative prc = bid/ask midpoint
        ret,
        vol,
        shrout
    FROM crsp.dsf
    WHERE date BETWEEN '{PRICE_START}' AND '{BACKTEST_END}'
      AND permno IN ({permnos_str})
""", date_cols=["date"])

price_raw["permno"] = price_raw["permno"].astype(int)
price_raw["mkt_cap"] = price_raw["prc"] * price_raw["shrout"] * 1000  # shrout is in thousands

print(f"Raw daily price records: {price_raw.shape[0]:,}")
print(f"Unique permnos: {price_raw['permno'].nunique()}")
print(f"Date range: {price_raw['date'].min().date()} → {price_raw['date'].max().date()}")

Raw daily price records: 2,934,733
Unique permnos: 871
Date range: 2008-01-02 → 2024-12-31


In [4]:
# ============================================================
# Cell 4: Build in_sp500 Boolean Mask (WITHOUT Filtering)
# ============================================================
# For each (permno, date), determine if the stock was an active S&P 500
# constituent on that date. This is the "dynamic constituent Boolean mask"
# required by Task 1.1.

price = price_raw.merge(sp500_members, on="permno", how="left")

# A stock is in the S&P 500 on date t if ANY of its membership windows
# contains that date
price["in_sp500"] = (
    (price["date"] >= price["start_date"]) &
    (price["date"] <= price["end_date"])
)

# A permno may have multiple membership windows → multiple rows per (permno, date).
# Collapse: keep the row, mark in_sp500 = True if ANY window matches.
price = (
    price
    .groupby(["permno", "date"], as_index=False)
    .agg({
        "prc": "first",
        "ret": "first",
        "vol": "first",
        "shrout": "first",
        "mkt_cap": "first",
        "in_sp500": "any"   # True if in ANY membership window
    })
)

# Assertion: (permno, date) must be unique after aggregation
assert not price.duplicated(["permno", "date"]).any(), \
    "FATAL: Duplicate (permno, date) rows detected after membership merge!"

print(f"Price panel shape: {price.shape[0]:,} rows × {price.shape[1]} cols")
print(f"in_sp500 coverage: {price['in_sp500'].mean():.2%} of all (permno, date) rows")

Price panel shape: 2,934,733 rows × 8 cols
in_sp500 coverage: 73.27% of all (permno, date) rows


In [5]:
# ============================================================
# Cell 5: Delist Returns & Adjusted Returns
# ============================================================
# Merge CRSP delisting data to construct ret_adj that accounts for
# delisting events (survivorship bias prevention).

delist = db.raw_sql(f"""
    SELECT
        permno,
        dlstdt,
        dlret,
        dlstcd
    FROM crsp.dsedelist
    WHERE permno IN ({permnos_str})
      AND dlstdt BETWEEN '{PRICE_START}' AND '{BACKTEST_END}'
""", date_cols=["dlstdt"])

delist["permno"] = delist["permno"].astype(int)

print(f"Delist events: {delist.shape[0]:,}")

# Merge delist info onto price panel
price = price.merge(
    delist,
    how="left",
    left_on=["permno", "date"],
    right_on=["permno", "dlstdt"]
)

# Construct adjusted return:
# On delisting day: ret_adj = (1 + ret) * (1 + dlret) - 1
# On all other days: ret_adj = ret
price["ret_adj"] = np.where(
    price["dlret"].notna(),
    (1 + price["ret"].fillna(0.0)) * (1 + price["dlret"].fillna(0.0)) - 1,
    price["ret"]
)

# Re-assert uniqueness after delist merge
assert not price.duplicated(["permno", "date"]).any(), \
    "FATAL: Duplicate (permno, date) rows after delist merge!"

print(f"ret_adj computed. Non-null dlret days: {price['dlret'].notna().sum():,}")

Delist events: 871
ret_adj computed. Non-null dlret days: 238


In [6]:
# ============================================================
# Cell 6: CRSP Security Names (dsenames)
# ============================================================
# Used for: security type screening (shrcd), exchange verification (exchcd),
# and ticker/company name lookups.

names = db.raw_sql(f"""
    SELECT
        permno,
        namedt,
        nameendt,
        shrcd,
        exchcd,
        ticker,
        ncusip,
        comnam,
        trdstat,
        secstat
    FROM crsp.dsenames
    WHERE permno IN ({permnos_str})
      AND namedt <= '{BACKTEST_END}'
      AND COALESCE(nameendt, '9999-12-31'::date) >= '{PRICE_START}'
""", date_cols=["namedt", "nameendt"])

names["nameendt"] = names["nameendt"].fillna(pd.Timestamp("2099-12-31"))
names["permno"] = names["permno"].astype(int)

print(f"dsenames records: {names.shape[0]:,}")
print(f"Unique permnos: {names['permno'].nunique()}")
print(f"shrcd distribution: {dict(names['shrcd'].value_counts().head(10))}")

dsenames records: 2,760
Unique permnos: 871
shrcd distribution: {np.int64(11): np.int64(2415), np.int64(12): np.int64(188), np.int64(18): np.int64(137), np.int64(48): np.int64(14), np.int64(71): np.int64(3), np.int64(72): np.int64(3)}


### Task 1.1 (cont.) — CRSP–Compustat Link Table (CCM)

In [7]:
# ============================================================
# Cell 7: CCM Link History (ALL link types)
# ============================================================
# Pull ALL link types (not just LC/LU) so we can apply priority-based
# deduplication. This fixes the earlier issue of missing links for
# some permnos.
#
# We also exclude any permnos with NO link at all (e.g., permno 16267).

ccm_all = db.raw_sql(f"""
    SELECT
        gvkey,
        lpermno AS permno,
        linktype,
        linkprim,
        linkdt,
        linkenddt
    FROM crsp.ccmxpf_lnkhist
    WHERE lpermno IN ({permnos_str})
""", date_cols=["linkdt", "linkenddt"])

ccm_all["permno"]    = pd.to_numeric(ccm_all["permno"], errors="coerce").astype("Int64")
ccm_all["gvkey"]     = ccm_all["gvkey"].astype(str).str.zfill(6)
ccm_all["linkenddt"] = ccm_all["linkenddt"].fillna(pd.Timestamp("2099-12-31"))

# Identify permnos with no CCM link at all
linked_permnos = set(ccm_all["permno"].dropna().astype(int).unique())
missing_permnos = sorted(set(permnos) - linked_permnos)
print(f"CCM records: {ccm_all.shape[0]:,}")
print(f"Linked permnos: {len(linked_permnos)}")
print(f"Permnos with NO CCM link (excluded from fundamentals): {len(missing_permnos)}")
if missing_permnos:
    print(f"  Missing: {missing_permnos}")

CCM records: 1,067
Linked permnos: 870
Permnos with NO CCM link (excluded from fundamentals): 1
  Missing: [16267]


In [8]:
# ============================================================
# Cell 8: CCM Priority-Based Deduplication
# ============================================================
# When a permno has multiple gvkey links, we rank them by quality:
#   linkprim: P (primary) > C > J > N
#   linktype: LC/LU (best) > LN/LS > LX/LD (weakest)
# This ensures 1:1 mapping for any given (permno, date).

PRIM_RANK = {"P": 0, "C": 1, "J": 2, "N": 3}
TYPE_RANK = {"LC": 0, "LU": 0, "LN": 1, "LS": 1, "LX": 2, "LD": 2}

ccm_all["prim_rank"] = ccm_all["linkprim"].map(PRIM_RANK).fillna(9).astype(int)
ccm_all["type_rank"] = ccm_all["linktype"].map(TYPE_RANK).fillna(9).astype(int)
ccm_all["link_rank"] = ccm_all["type_rank"] * 10 + ccm_all["prim_rank"]

print(f"Link rank distribution:\n{ccm_all['link_rank'].value_counts().sort_index()}")

Link rank distribution:
link_rank
0     872
1     154
2      14
3       4
10      2
11      2
20      3
21      1
22      1
23     14
Name: count, dtype: int64


### Task 1.2 — Quarterly Fundamentals with PIT Discipline

In [9]:
# ============================================================
# Cell 9: Quarterly Fundamentals from Compustat
# ============================================================
# FIX: Added popsrc='D' (domestic population source) which was missing
# in the previous rebuild version.

gvkeys = sorted(ccm_all["gvkey"].dropna().unique().tolist())
gvkeys_str = ",".join([f"'{g}'" for g in gvkeys])

fundq = db.raw_sql(f"""
    SELECT
        gvkey,
        datadate,
        fyearq,
        fqtr,
        rdq,
        epspiq,        -- EPS (including extraordinary items)
        niq,           -- Net Income
        ibq,           -- Income Before Extraordinary Items
        ceqq,          -- Common/Ordinary Equity (Book Value)
        prccq,         -- Price Close - Quarter End
        cshoq,         -- Common Shares Outstanding
        atq,           -- Total Assets
        ltq            -- Total Liabilities
    FROM comp.fundq
    WHERE gvkey IN ({gvkeys_str})
      AND datadate BETWEEN '{FUNDQ_START}' AND '{BACKTEST_END}'
      AND indfmt  = 'INDL'
      AND datafmt = 'STD'
      AND popsrc  = 'D'
      AND consol  = 'C'
""", date_cols=["datadate", "rdq"])

print(f"Raw fundq records: {fundq.shape[0]:,}, gvkeys: {fundq['gvkey'].nunique()}")
print(f"rdq missing: {fundq['rdq'].isna().sum()} ({fundq['rdq'].isna().mean():.2%})")

Raw fundq records: 55,101, gvkeys: 866
rdq missing: 1116 (2.03%)


In [10]:
# ============================================================
# Cell 10: Point-in-Time (PIT) Date Construction
# ============================================================
# Per Proposal Task 1.2:
#   "Strictly discard the use of natural quarter-end dates; all alignments
#    must be based on the actual disclosure date [...] artificially
#    incorporating a 45–90 day filing delay."
#
# Implementation:
#   pit_date = max(rdq + 1 day, datadate + MIN_DELAY_FROM_DATADATE)
#
# This ensures:
#   1) Data is never used before it is publicly announced (rdq)
#   2) A minimum of 45 days from fiscal period end (conservative buffer)

# Drop rows with missing rdq (cannot determine announcement date)
n_before = len(fundq)
fundq = fundq.dropna(subset=["rdq"]).copy()
print(f"Dropped {n_before - len(fundq)} rows with missing rdq")

# Construct PIT date
fundq["pit_date"] = pd.concat([
    fundq["rdq"] + pd.Timedelta(days=1),
    fundq["datadate"] + pd.Timedelta(days=MIN_DELAY_FROM_DATADATE)
], axis=1).max(axis=1)

# ---- Code-Level Anti-Look-Ahead Assertions (Task 1.2) ----

# 1) PIT date must never be before fiscal period end
assert (fundq["pit_date"] >= fundq["datadate"]).all(), \
    "FATAL: pit_date < datadate — potential look-ahead bias!"

# 2) PIT date must be after report date
assert (fundq["pit_date"] > fundq["rdq"]).all(), \
    "FATAL: pit_date <= rdq — using data before public announcement!"

# 3) Sanity check: PIT lag statistics
pit_lag = (fundq["pit_date"] - fundq["datadate"]).dt.days
print(f"PIT lag (pit_date - datadate) statistics:")
print(f"  min={pit_lag.min()}, median={pit_lag.median():.0f}, "
      f"mean={pit_lag.mean():.0f}, max={pit_lag.max()}")
assert pit_lag.min() >= MIN_DELAY_FROM_DATADATE, \
    f"FATAL: Minimum PIT lag ({pit_lag.min()}) < {MIN_DELAY_FROM_DATADATE} days!"

print(f"\nPIT fundq: {fundq.shape[0]:,} rows, {fundq['gvkey'].nunique()} gvkeys")
print(f"datadate range: {fundq['datadate'].min().date()} → {fundq['datadate'].max().date()}")
print(f"pit_date range: {fundq['pit_date'].min().date()} → {fundq['pit_date'].max().date()}")

Dropped 1116 rows with missing rdq
PIT lag (pit_date - datadate) statistics:
  min=45, median=45, mean=46, max=640

PIT fundq: 53,985 rows, 866 gvkeys
datadate range: 2006-01-31 → 2024-12-31
pit_date range: 2006-03-17 → 2025-03-18


In [11]:
# ============================================================
# Cell 11: GICS Sector Classification
# ============================================================
# Static sector assignment from comp.company.
# Used for intra-sector standardization (Task 1.3) and sector neutrality
# constraints (Task 3.3).

company = db.raw_sql("""
    SELECT gvkey, gsector, gind, gsubind
    FROM comp.company
""")
company["gvkey"] = company["gvkey"].astype(str).str.zfill(6)

fundq = fundq.merge(company[["gvkey", "gsector"]], on="gvkey", how="left")

gsector_miss = fundq["gsector"].isna().mean()
print(f"GICS sector merge — missing rate: {gsector_miss:.4%}")
if gsector_miss > 0.01:
    print("WARNING: >1% of fundq rows missing gsector!")

GICS sector merge — missing rate: 0.0000%


In [12]:
# ============================================================
# Cell 12: Merge Fundamentals with CCM to Get permno
# ============================================================
# Use the priority-ranked CCM link table to map gvkey → permno.
# For each (gvkey, pit_date), keep only the best-ranked link that
# is valid during the pit_date window.

tmp = fundq.merge(ccm_all, on="gvkey", how="left")

# Filter: pit_date must fall within the link's valid window
tmp = tmp[
    (tmp["pit_date"] >= tmp["linkdt"]) &
    (tmp["pit_date"] <= tmp["linkenddt"])
].copy()

# Deduplicate: keep the best-ranked link (lowest link_rank, most recent linkdt)
tmp = (
    tmp.sort_values(
        ["gvkey", "pit_date", "link_rank", "linkdt"],
        ascending=[True, True, True, False]
    )
    .drop_duplicates(["gvkey", "pit_date"], keep="first")
)

tmp["permno"] = tmp["permno"].astype("Int64")

# QC
permno_miss = tmp["permno"].isna().mean()
print(f"fundq-CCM merge: {tmp.shape[0]:,} rows, {tmp['permno'].nunique()} unique permnos")
print(f"Missing permno rate: {permno_miss:.4%}")

# Drop rows without valid permno link
fundq_linked = tmp.dropna(subset=["permno"]).copy()
print(f"After dropping unlinked: {fundq_linked.shape[0]:,} rows")

fundq-CCM merge: 52,034 rows, 863 unique permnos
Missing permno rate: 0.0000%
After dropping unlinked: 52,034 rows


---
## Phase 1.3 + Phase 2.1: Factor Signal Construction

### Task 1.3 — MAD Winsorization & Intra-Sector Standardization
### Task 2.1 — Base Factor Exposure Calculation

In [13]:
# ============================================================
# Cell 13: Month-End Grid & PIT Merge (merge_asof)
# ============================================================
# Build a month-end snapshot for each (permno, month) by taking the
# last trading day of each month. Then align PIT fundamentals using
# backward merge_asof.

p = price.sort_values(["permno", "date"]).copy()
p["month"] = p["date"].dt.to_period("M")

# Take the last trading day of each month for each permno
month_end = (
    p.groupby(["permno", "month"], as_index=False)
    .tail(1)[["permno", "date", "mkt_cap", "prc", "in_sp500"]]
)

# merge_asof requires exact dtype match for both `by` and `on` keys.
month_end["permno"] = pd.to_numeric(month_end["permno"], errors="coerce")
month_end = month_end.dropna(subset=["permno"]).copy()
month_end["permno"] = month_end["permno"].astype("int64")
month_end["date"] = pd.to_datetime(month_end["date"])

# Prepare the factor source table for merge_asof (and normalize dtypes)
fac_src = fundq_linked.dropna(subset=["permno", "pit_date"]).copy()
fac_src["permno"] = pd.to_numeric(fac_src["permno"], errors="coerce")
fac_src = fac_src.dropna(subset=["permno"]).copy()
fac_src["permno"] = fac_src["permno"].astype("int64")
fac_src["pit_date"] = pd.to_datetime(fac_src["pit_date"])

# Sort for merge_asof requirement
month_end = month_end.sort_values(["date", "permno"]).reset_index(drop=True)
fac_src = fac_src.sort_values(["pit_date", "permno"]).reset_index(drop=True)

# Backward merge: for each (permno, date), get the most recent record
# with pit_date <= date. This strictly enforces PIT discipline.
sig = pd.merge_asof(
    month_end,
    fac_src,
    left_on="date",
    right_on="pit_date",
    by="permno",
    direction="backward"
)

# QC: what fraction of month-end rows have no matched fundamentals?
miss_rate = sig["gvkey"].isna().mean()
print(f"Month-end rows: {sig.shape[0]:,}")
print(f"Fundamental match miss rate: {miss_rate:.2%}")
print(f"Permnos with zero fundamental matches: "
      f"{(sig['gvkey'].isna()).groupby(sig['permno']).all().sum()}")

Month-end rows: 140,070
Fundamental match miss rate: 0.72%
Permnos with zero fundamental matches: 8


In [14]:
# ============================================================
# Cell 14: MAD Winsorized Z-Score Function
# ============================================================
# Per Proposal Task 1.3:
#   "Apply the Median Absolute Deviation (MAD) method for cross-sectional
#    Winsorization. All Z-score standardization calculations must be strictly
#    executed within GICS sector groupings."
#
# Implementation:
#   1) Compute robust z = (x - median) / (1.4826 * MAD)
#   2) Winsorize (clip) to [-n_mad, +n_mad]
#   3) Re-standardize to mean=0, std=1

def mad_winsorized_zscore(x, n_mad=5.0):
    """
    MAD-based winsorization followed by standard z-score normalization.

    Parameters
    ----------
    x : pd.Series
        Raw cross-sectional values within a single (date, gsector) group.
    n_mad : float
        Winsorization threshold in MAD units (default: 5.0).

    Returns
    -------
    pd.Series : Winsorized, standardized z-scores.
    """
    x = x.astype(float)
    valid = x.notna()

    if valid.sum() < 5:
        # Too few observations for meaningful standardization
        return pd.Series(np.nan, index=x.index)

    x_valid = x[valid]
    median = x_valid.median()
    mad = (x_valid - median).abs().median()

    if mad == 0 or not np.isfinite(mad):
        # No dispersion → all values effectively identical
        return pd.Series(0.0, index=x.index).where(valid, np.nan)

    # Step 1: Robust z-score (MAD-based)
    robust_z = (x - median) / (1.4826 * mad)

    # Step 2: Winsorize to [-n_mad, +n_mad]
    robust_z_clipped = robust_z.clip(-n_mad, n_mad)

    # Step 3: Re-standardize to mean=0, std=1
    z_mean = robust_z_clipped[valid].mean()
    z_std  = robust_z_clipped[valid].std(ddof=0)

    if z_std == 0 or not np.isfinite(z_std):
        return pd.Series(0.0, index=x.index).where(valid, np.nan)

    z = (robust_z_clipped - z_mean) / z_std
    z[~valid] = np.nan
    return z

# Quick unit test
test = pd.Series([1, 2, 3, 4, 5, 100, np.nan])
result = mad_winsorized_zscore(test)
print("MAD winsorized z-score unit test:")
print(f"  Input:  {test.tolist()}")
print(f"  Output: {[round(v, 3) if pd.notna(v) else None for v in result]}")
print(f"  Max absolute z: {result.abs().max():.3f} (should be clipped)")

MAD winsorized z-score unit test:
  Input:  [1.0, 2.0, 3.0, 4.0, 5.0, 100.0, nan]
  Output: [-0.871, -0.65, -0.429, -0.207, 0.014, 2.143, None]
  Max absolute z: 2.143 (should be clipped)


In [16]:
# ============================================================
# Cell 15: Compute Raw Metrics & Apply Winsorized Z-Scores
# ============================================================
# Earnings Yield and Price-to-Book per Proposal Task 2.1.
#
# Definitions:
#   earnings_yield = epspiq / |prccq|   (EPS / Price)
#   price_to_book  = |prccq| / (ceqq / cshoq)   (Price / Book per Share)
#
# Note: prccq from Compustat can be negative (same CRSP convention);
# we take absolute value for consistency.

# Coerce to numeric and clean
for col in ["epspiq", "prccq", "ceqq", "cshoq"]:
    sig[col] = pd.to_numeric(sig[col], errors="coerce")

sig["prccq_abs"] = sig["prccq"].abs()

# Earnings Yield: EPS / |Price|
# Guard: only compute where price is positive
mask_ey = sig["prccq_abs"].gt(0) & sig["epspiq"].notna()
sig["earnings_yield"] = np.nan
sig.loc[mask_ey, "earnings_yield"] = (
    sig.loc[mask_ey, "epspiq"] / sig.loc[mask_ey, "prccq_abs"]
)

# Price-to-Book: |Price| / (Book Equity / Shares Outstanding)
# Guard: ceqq > 0 and cshoq > 0
mask_pb = (
    sig["prccq_abs"].gt(0) &
    sig["ceqq"].gt(0) &
    sig["cshoq"].gt(0)
)
sig["book_per_share"] = np.nan
sig.loc[mask_pb, "book_per_share"] = (
    sig.loc[mask_pb, "ceqq"] / sig.loc[mask_pb, "cshoq"]
)
sig["price_to_book"] = np.nan
sig.loc[mask_pb, "price_to_book"] = (
    sig.loc[mask_pb, "prccq_abs"] / sig.loc[mask_pb, "book_per_share"]
)

# Log market cap for Size factor
sig["log_mktcap"] = np.log(sig["mkt_cap"].replace(0, np.nan))

print(f"earnings_yield  valid: {sig['earnings_yield'].notna().sum():,} "
      f"({sig['earnings_yield'].notna().mean():.2%})")
print(f"price_to_book   valid: {sig['price_to_book'].notna().sum():,} "
      f"({sig['price_to_book'].notna().mean():.2%})")
print(f"log_mktcap      valid: {sig['log_mktcap'].notna().sum():,} "
      f"({sig['log_mktcap'].notna().mean():.2%})")

earnings_yield  valid: 138,618 (98.96%)
price_to_book   valid: 132,700 (94.74%)
log_mktcap      valid: 139,842 (99.84%)


/usr/local/sas/jupyterhub/prod/venvs/20250514/lib/python3.13/site-packages/pandas/core/arrays/masked.py:672: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs2, **kwargs)


In [17]:
# ============================================================
# Cell 16: Intra-Sector MAD Winsorized Z-Scores (Task 1.3)
# ============================================================
# Per Proposal: "All Z-score standardization calculations must be strictly
# executed within GICS sector groupings."
#
# This applies to ALL three factor components:
#   - earnings_yield → ey_z   (within date × gsector)
#   - price_to_book  → pb_z   (within date × gsector)
#   - log_mktcap     → size_z (within date × gsector)  ← FIX: was full cross-section before
#
# The grouping key is (date, gsector) where date is the month-end date.

# Convert gsector to numeric for clean grouping
sig["gsector"] = pd.to_numeric(sig["gsector"], errors="coerce")

group_keys = ["date", "gsector"]

print("Computing intra-sector MAD winsorized z-scores...")
for raw_col, z_col in [
    ("earnings_yield", "ey_z"),
    ("price_to_book",  "pb_z"),
    ("log_mktcap",     "size_z"),
]:
    sig[z_col] = (
        sig.groupby(group_keys, group_keys=False)[raw_col]
          .apply(mad_winsorized_zscore)
    )
    extremes = sig[z_col].abs().describe()
    print(f"  {z_col:8s}: mean={sig[z_col].mean():.4f}, "
          f"std={sig[z_col].std():.4f}, "
          f"min={sig[z_col].min():.4f}, max={sig[z_col].max():.4f}")

Computing intra-sector MAD winsorized z-scores...
  ey_z    : mean=-0.0000, std=1.0000, min=-4.2832, max=4.3554
  pb_z    : mean=-0.0000, std=1.0000, min=-2.7442, max=4.2996
  size_z  : mean=-0.0000, std=1.0000, min=-4.4378, max=3.5628


In [18]:
# ============================================================
# Cell 17: Factor Composite Construction (Task 2.1)
# ============================================================
# Per Proposal:
#   Value(i, t) = 0.67 * Z(earnings_yield) + 0.33 * Z(price_to_book)
#   Size(i, t)  = - Z(market_cap)
#
# NOTE on Value formula sign convention:
#   The proposal specifies a POSITIVE coefficient on Z(P/B). This means
#   higher P/B contributes positively to the value score. If the intent
#   was to favor low-P/B (deep value), the sign should be negative.
#   We implement the formula EXACTLY as written in the proposal.

sig["value_factor"] = 0.67 * sig["ey_z"] + 0.33 * sig["pb_z"]
sig["size_factor"]  = -sig["size_z"]

print("Factor statistics:")
for fac in ["value_factor", "size_factor"]:
    s = sig[fac].dropna()
    print(f"  {fac:15s}: N={len(s):,}, mean={s.mean():.4f}, "
          f"std={s.std():.4f}, min={s.min():.4f}, max={s.max():.4f}")

# Verify winsorization worked: extreme z-scores should be bounded
for z_col in ["ey_z", "pb_z", "size_z"]:
    max_abs = sig[z_col].abs().max()
    print(f"  {z_col} max |z|: {max_abs:.2f}", end="")
    if max_abs > 10:
        print(" ← WARNING: still has extreme values")
    else:
        print(" ✓")

Factor statistics:
  value_factor   : N=132,469, mean=0.0080, std=0.7006, min=-3.2578, max=3.2010
  size_factor    : N=138,633, mean=0.0000, std=1.0000, min=-3.5628, max=4.4378
  ey_z max |z|: 4.36 ✓
  pb_z max |z|: 4.30 ✓
  size_z max |z|: 4.44 ✓


### Monthly Return Aggregation
Needed for Phase 2 (cross-sectional regression) and Phase 4 (performance attribution).

In [19]:
# ============================================================
# Cell 18: Monthly Return Aggregation
# ============================================================
# Compute monthly returns from daily ret_adj by compounding within
# each (permno, month). This is needed for:
#   - Task 2.2: Cross-sectional regression (excess stock returns)
#   - Task 4.1: NAV simulation
#   - Momentum factor construction (cumulative 12-1 month returns)

p_ret = price[["permno", "date", "ret_adj", "in_sp500"]].copy()
p_ret["month"] = p_ret["date"].dt.to_period("M")

# Compound daily returns within each month
def compound_returns(rets):
    """Compound a series of simple returns: prod(1+r) - 1."""
    return (1 + rets.fillna(0)).prod() - 1

monthly_ret = (
    p_ret
    .groupby(["permno", "month"], as_index=False)
    .agg(
        ret_monthly=("ret_adj", compound_returns),
        n_days=("ret_adj", "size"),
        last_date=("date", "max"),
        in_sp500_eom=("in_sp500", "last")  # S&P 500 status at month end
    )
)

monthly_ret["ret_monthly"] = monthly_ret["ret_monthly"].clip(-0.95, 2.0)

print(f"Clipped to [-95%, +200%]. "
      f"Affected rows: {((monthly_ret['ret_monthly'] == -0.95) | (monthly_ret['ret_monthly'] == 2.0)).sum()}")

print(f"Monthly returns: {monthly_ret.shape[0]:,} rows")
print(f"Date range: {monthly_ret['month'].min()} → {monthly_ret['month'].max()}")
print(f"ret_monthly stats:\n{monthly_ret['ret_monthly'].describe()}")

Clipped to [-95%, +200%]. Affected rows: 10
Monthly returns: 140,070 rows
Date range: 2008-01 → 2024-12
ret_monthly stats:
count    140070.000000
mean          0.011661
std           0.108235
min          -0.950000
25%          -0.041101
50%           0.011597
75%           0.062305
max           2.000000
Name: ret_monthly, dtype: float64


### Data Quality Checks & Assertions

In [20]:
# ============================================================
# Cell 19: Comprehensive QC Checks
# ============================================================

print("=" * 60)
print("  COMPREHENSIVE DATA QUALITY CHECKS")
print("=" * 60)

# 1) Price panel uniqueness
assert not price.duplicated(["permno", "date"]).any(), "Duplicate (permno, date)!"
print("✓ Price panel: (permno, date) uniqueness verified")

# 2) PIT discipline
assert (fundq["pit_date"] >= fundq["datadate"] + pd.Timedelta(days=MIN_DELAY_FROM_DATADATE)).all(), \
    "PIT delay below minimum!"
print(f"✓ PIT discipline: all records have >= {MIN_DELAY_FROM_DATADATE}-day delay")

# 3) No future data leakage in signals
sig_check = sig.dropna(subset=["pit_date"])
assert (sig_check["pit_date"] <= sig_check["date"]).all(), \
    "FATAL: Signal uses future data (pit_date > month-end date)!"
print("✓ No look-ahead bias: all pit_date <= month-end date")



# 4) Z-score winsorization effectiveness
for z_col in ["ey_z", "pb_z", "size_z"]:
    max_abs = sig[z_col].abs().max()
    # After MAD winsorization + re-standardization, values should be bounded
    assert max_abs < 20, f"{z_col} has extreme values ({max_abs:.1f})"
print("✓ Z-score winsorization: all factors bounded (max |z| < 20)")

# 5) Cross-file consistency
price_permnos = set(price["permno"].unique())
sig_final_permnos = set(sig.dropna(subset=["gvkey"])["permno"].astype(int).unique())
ccm_permnos   = set(ccm_all["permno"].dropna().astype(int).unique())
print(f"✓ permno overlap: price∩ccm = {len(price_permnos & ccm_permnos)}, "
      f"sig(with fundamentals)⊂ccm = {sig_final_permnos.issubset(ccm_permnos)}")

# 6) Sector coverage in signals
n_sectors = sig.dropna(subset=["gsector"])["gsector"].nunique()
print(f"✓ GICS sectors in signals: {n_sectors} (expected 11)")

# 7) Monthly return coverage
print(f"✓ Monthly returns: {monthly_ret.shape[0]:,} obs, "
      f"{monthly_ret['permno'].nunique()} permnos")

print("\n" + "=" * 60)
print("  ALL CHECKS PASSED")
print("=" * 60)

  COMPREHENSIVE DATA QUALITY CHECKS
✓ Price panel: (permno, date) uniqueness verified
✓ PIT discipline: all records have >= 45-day delay
✓ No look-ahead bias: all pit_date <= month-end date
✓ Z-score winsorization: all factors bounded (max |z| < 20)
✓ permno overlap: price∩ccm = 870, sig(with fundamentals)⊂ccm = True
✓ GICS sectors in signals: 11 (expected 11)
✓ Monthly returns: 140,070 obs, 871 permnos

  ALL CHECKS PASSED


---
## Export: Save All Datasets

In [21]:
# ============================================================
# Cell 20: Filter Final Signal Panel
# ============================================================
# Keep only rows with valid fundamentals and within backtest period

sig_final = sig[
    sig["earnings_yield"].notna() &
    sig["price_to_book"].notna() &
    sig["gvkey"].notna() &
    sig["gsector"].notna() &
    (sig["date"] >= BACKTEST_START) &
    (sig["date"] <= BACKTEST_END)
].copy()

cols_to_keep = [
    "permno", "date", "gvkey", "gsector",
    "mkt_cap",
    "earnings_yield", "price_to_book",
    "ey_z", "pb_z", "size_z",
    "value_factor", "size_factor"
]
sig_final = sig_final[cols_to_keep].copy()

# Convert gsector to int for clean output
sig_final["gsector"] = sig_final["gsector"].astype(int)

print(f"Final signal panel: {sig_final.shape[0]:,} rows × {sig_final.shape[1]} cols")
print(f"Unique permnos: {sig_final['permno'].nunique()}")
print(f"Date range: {sig_final['date'].min().date()} → {sig_final['date'].max().date()}")
print(f"Sectors: {sorted(sig_final['gsector'].unique())}")

Final signal panel: 115,962 rows × 12 cols
Unique permnos: 822
Date range: 2010-01-26 → 2024-12-31
Sectors: [np.int64(10), np.int64(15), np.int64(20), np.int64(25), np.int64(30), np.int64(35), np.int64(40), np.int64(45), np.int64(50), np.int64(55), np.int64(60)]


In [22]:
# ============================================================
# Cell 21: Save Price Panel (with delist adjustments)
# ============================================================
# This is the core daily panel used throughout all phases.
# Contains ALL trading days (not filtered by in_sp500) to support
# lookback windows for momentum, rolling beta, and covariance.

price_cols = [
    "permno", "date", "prc", "ret", "vol", "shrout", "mkt_cap",
    "in_sp500", "dlstdt", "dlret", "dlstcd", "ret_adj"
]
price_out = price[price_cols].copy()

# Save as parquet (compact, fast) and csv.gz (portable)
price_out.to_parquet("sp500_price_panel_with_delist.parquet", index=False)
price_out.to_csv("sp500_price_panel_with_delist.csv.gz", index=False, compression="gzip")

print(f"Saved price panel: {price_out.shape[0]:,} rows")
print(f"  in_sp500=True:  {price_out['in_sp500'].sum():,}")
print(f"  in_sp500=False: {(~price_out['in_sp500']).sum():,}")

Saved price panel: 2,934,733 rows
  in_sp500=True:  2,150,329
  in_sp500=False: 784,404


In [23]:
# ============================================================
# Cell 22: Save Supporting Tables
# ============================================================

# 1) CCM link history (all types, with priority ranks)
ccm_export = ccm_all.drop(columns=["prim_rank", "type_rank", "link_rank"], errors="ignore")
ccm_export.to_csv("sp500_ccm_lnkhist_all_nodvmt.csv.gz", index=False, compression="gzip")
print(f"Saved CCM link table: {ccm_export.shape[0]:,} rows")

# 2) dsenames
names.to_csv("sp500_dsenames_csv.gz", index=False, compression="gzip")
print(f"Saved dsenames: {names.shape[0]:,} rows")

# 3) Quarterly fundamentals (PIT) — without linktype/rank columns
fundq_export_cols = [
    "gvkey", "datadate", "fyearq", "fqtr", "rdq",
    "epspiq", "niq", "ibq", "ceqq", "prccq", "cshoq",
    "atq", "ltq", "pit_date"
]
fundq_export = fundq[fundq_export_cols].copy()
fundq_export.to_csv("sp500_fundq_pit_rebuilt_nodvmt.csv.gz", index=False, compression="gzip")
print(f"Saved fundq PIT: {fundq_export.shape[0]:,} rows")

# 4) Monthly returns
monthly_ret.to_csv("sp500_monthly_returns.csv.gz", index=False, compression="gzip")
print(f"Saved monthly returns: {monthly_ret.shape[0]:,} rows")

Saved CCM link table: 1,067 rows
Saved dsenames: 2,760 rows
Saved fundq PIT: 53,985 rows
Saved monthly returns: 140,070 rows


In [24]:
# ============================================================
# Cell 23: Save Monthly Signal Panel
# ============================================================

sig_final.to_parquet("sp500_monthly_signals_rebuilt_nodvmt.parquet", index=False)
sig_final.to_csv("sp500_monthly_signals_rebuilt_nodvmt.csv.gz", index=False, compression="gzip")

print(f"Saved monthly signals: {sig_final.shape[0]:,} rows × {sig_final.shape[1]} cols")

Saved monthly signals: 115,962 rows × 12 cols


In [25]:
# ============================================================
# Cell 24: Final Summary
# ============================================================

print("=" * 70)
print("  DATA PIPELINE COMPLETE — OUTPUT FILE SUMMARY")
print("=" * 70)

import os

output_files = [
    ("sp500_price_panel_with_delist.parquet", "Daily price panel (parquet)"),
    ("sp500_price_panel_with_delist.csv.gz",  "Daily price panel (csv.gz)"),
    ("sp500_ccm_lnkhist_all_nodvmt.csv.gz",   "CCM link history"),
    ("sp500_dsenames_csv.gz",                  "CRSP security names"),
    ("sp500_fundq_pit_rebuilt_nodvmt.csv.gz",  "Quarterly fundamentals (PIT)"),
    ("sp500_monthly_returns.csv.gz",           "Monthly compounded returns"),
    ("sp500_monthly_signals_rebuilt_nodvmt.parquet", "Monthly factor signals (parquet)"),
    ("sp500_monthly_signals_rebuilt_nodvmt.csv.gz",  "Monthly factor signals (csv.gz)"),
]

for fname, desc in output_files:
    if os.path.exists(fname):
        size_mb = os.path.getsize(fname) / 1e6
        print(f"  ✓ {fname:50s} {size_mb:8.1f} MB  — {desc}")
    else:
        print(f"  ✗ {fname:50s}  MISSING  — {desc}")

print()
print("Corrections applied in this version:")
print("  [1] Value factor: 0.67*Z(EY) + 0.33*Z(P/B) per proposal (was 0.5/0.5 with flipped P/B sign)")
print("  [2] Size factor: intra-sector z-score (was full cross-section)")
print("  [3] MAD winsorization restored (was removed in previous rebuild)")
print("  [4] Price panel retains all dates (in_sp500 is flag, not filter)")
print("  [5] fundq SQL includes popsrc='D' (was missing)")
print("  [6] (permno, date) uniqueness asserted after every merge")
print("  [7] Earnings yield definition: epspiq/|prccq| (consistent throughout)")
print("  [8] PIT delay: max(rdq+1, datadate+45d) within proposal 45-90d range")
print("  [9] Single linear pipeline (duplicate code paths removed)")
print("  [10] prccq takes absolute value (Compustat negative price convention)")
print("  [11] Monthly return aggregation added (compound daily ret_adj)")

print("\n" + "=" * 70)
print("  Ready for Phase 2 (FRP), Phase 3 (Optimization), Phase 4 (Backtest)")
print("=" * 70)

  DATA PIPELINE COMPLETE — OUTPUT FILE SUMMARY
  ✓ sp500_price_panel_with_delist.parquet                  67.4 MB  — Daily price panel (parquet)
  ✓ sp500_price_panel_with_delist.csv.gz                   68.0 MB  — Daily price panel (csv.gz)
  ✓ sp500_ccm_lnkhist_all_nodvmt.csv.gz                     0.0 MB  — CCM link history
  ✓ sp500_dsenames_csv.gz                                   0.0 MB  — CRSP security names
  ✓ sp500_fundq_pit_rebuilt_nodvmt.csv.gz                   1.7 MB  — Quarterly fundamentals (PIT)
  ✓ sp500_monthly_returns.csv.gz                            2.2 MB  — Monthly compounded returns
  ✓ sp500_monthly_signals_rebuilt_nodvmt.parquet            7.4 MB  — Monthly factor signals (parquet)
  ✓ sp500_monthly_signals_rebuilt_nodvmt.csv.gz             8.6 MB  — Monthly factor signals (csv.gz)

Corrections applied in this version:
  [1] Value factor: 0.67*Z(EY) + 0.33*Z(P/B) per proposal (was 0.5/0.5 with flipped P/B sign)
  [2] Size factor: intra-sector z-score (was ful